#### CARGA INCREMENTAL

CONFIGURACION

In [0]:

CATALOG = "pentaho_logs"
SCHEMA = "bronze"
TABLE = "bronze_logs_carte"
VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_cartelogs"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

#### OBTENCION DE NOMBRE POR VOLUMEN

In [0]:
if dbutils.fs.ls(VOLUME_PATH):

    archivos_volumen = [
        archivo.name
        for archivo in dbutils.fs.ls(VOLUME_PATH)
        if archivo.name.endswith(".log")
    ]
    for archivo in archivos_volumen:
        print(archivo)

    display(
        spark.createDataFrame(
            [(archivo,) for archivo in archivos_volumen],
            ["file_name"]
        )
    )

else:
    print("No hay archivos en el Volume")
    archivos_volumen = []

#### OBTENCIÓN DE NOMBRE POR TABLA


In [0]:
if spark.catalog.tableExists(TABLE_NAME):
 print("La tabla bronze existe")
 archivos_bronze = [
       row.file_name
        for row in (spark.table(TABLE_NAME).select("file_name").distinct().collect())
    ]
 
 print("=== ARCHIVOS EN BRONZE ===")
 for archivo in archivos_bronze:
   print(archivo)
else:
 archivos_bronze = []

#### COMPARAR VOLUMEN VS TABLA BRONZE

In [0]:
## comparamos los archivos en un arreglo si hay archivos en donde sean diferentes ingresa

archivos_nuevos = [
    archivo
    for archivo in archivos_volumen
    if archivo not in archivos_bronze
]

print("=== NUEVOS ===")
print(archivos_nuevos)

#### envio de parametro para capa BRONZE

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/bronze/bronze_carte/01_Bronze_Ingestion_Carte",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")

#### Envio de parametro para Capa Silver

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/silver/silver_carte/01_Silver_Transformation_Carte",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")




    

#### Envio de parametro para Capa Gold

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/gold/gold_carte/01_Gold_log_Carte",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")